In [2]:

import tkinter as tk
from tkinter import ttk
from decimal import Decimal
from decimal import Decimal, ROUND_HALF_UP
from tkinter import messagebox
from PIL import Image, ImageTk
import mysql.connector

def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="touqeer@rif311030103100",   
        database="model"
    )

def save_user(username, password, phoneno, address, email):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute(
            "INSERT INTO users (name, password, phone_no, address, email) VALUES (%s, %s, %s, %s, %s)",
            (username, password, phoneno, address, email) 
        )
        conn.commit()
        return True
    except mysql.connector.IntegrityError:
        return False
    finally:
        cursor.close()
        conn.close()

def check_login(username, password):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users WHERE name=%s AND password=%s", (username, password))
    user = cursor.fetchone()
    cursor.close()
    conn.close()
    return user is not None

def check_admin(username, password):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT * FROM admin WHERE name=%s AND password=%s", (username, password))
        admin = cursor.fetchone()
        return admin is not None
    finally:
        cursor.close()
        conn.close()
        
def get_user_balance(user_id):
    conn = get_connection()
    cursor=conn.cursor()
    cursor.execute("SELECT balance FROM users WHERE id=%s", (user_id,))
    result = cursor.fetchone()
    conn.close()
    return result[0] if result else None

def update_user_balance(user_id, new_balance):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE users SET balance=%s WHERE id=%s", (new_balance, user_id))
    conn.commit()
    conn.close()
    
def fetch_categories():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT id, name FROM category")
    categories = cursor.fetchall()
    cursor.close()
    conn.close()
    return categories

def fetch_products_by_category(category_id):
    conn = get_connection()
    cursor = conn.cursor(dictionary=True)
    cursor.execute("SELECT name, price, stock FROM products WHERE category_id=%s", (category_id,))
    products = cursor.fetchall()
    cursor.close()
    conn.close()
    return products
    
def fetch_users():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT id, name, email, password, phone_no, address, balance FROM users")
    users = cursor.fetchall()
    cursor.close()
    conn.close()
    return users

def save_order(user_id, cart_items, total):
    conn = get_connection()
    cursor = conn.cursor()
    for item in cart_items:
        cursor.execute(
            "INSERT INTO orders (user_id, product_name, price, quantity, total, status) VALUES (%s, %s, %s, %s, %s, %s)",
            (user_id, item["name"], item["price"], 1, item["price"], "Completed")  
        )
    conn.commit()
    cursor.close()
    conn.close()

def edit_user(user_id, name, email, password, phone_no, address, balance):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE users SET name=%s, email=%s, password=%s, phone_no=%s, address=%s, balance=%s WHERE id=%s",
        (name, email, password, phone_no, address, balance, user_id)
    )
    conn.commit()
    cursor.close()
    conn.close()

def delete_user(user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM users WHERE id=%s", (user_id,))
    conn.commit()
    cursor.close()
    conn.close()


def fetch_orders():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT o.id, u.name, o.total, o.order_date, o.status
        FROM orders o
        JOIN users u ON o.user_id = u.id
    """)
    orders = cursor.fetchall()
    cursor.close()
    conn.close()
    return orders

def view_users(parent):
    users = fetch_users()
    win = tk.Toplevel(parent)
    win.title("All Users")
    center_window(win, 1000, 600)
    win.configure(bg="#f0f4c3")

    tk.Label(win, text="Users List", font=("Helvetica", 20, "bold"), bg="#f0f4c3").pack(pady=10)

    container = tk.Frame(win)
    container.pack(fill="both", expand=True)

    canvas = tk.Canvas(container, bg="#f0f4c3", highlightthickness=0)
    v_scroll = tk.Scrollbar(container, orient="vertical", command=canvas.yview)
    h_scroll = tk.Scrollbar(container, orient="horizontal", command=canvas.xview)

    scroll_frame = tk.Frame(canvas, bg="#f0f4c3")

    scroll_frame.bind(
        "<Configure>",
        lambda e: canvas.configure(
            scrollregion=canvas.bbox("all")
        )
    )

    canvas.create_window((0, 0), window=scroll_frame, anchor="nw")
    canvas.configure(yscrollcommand=v_scroll.set, xscrollcommand=h_scroll.set)

    canvas.pack(side="left", fill="both", expand=True)
    v_scroll.pack(side="right", fill="y")
    h_scroll.pack(side="bottom", fill="x")

    for user in users:
        user_id, name, email, pwd, phoneno, address, balance= user

        frame = tk.Frame(scroll_frame, bg="white", bd=2, relief="groove")
        frame.pack(fill="x", padx=20, pady=5)

        tk.Label(
            frame,
            text=(
                f"ID: {user_id}\n"
                f"Name: {name}\n"
                f"Email: {email}\n"
                f"Password: {pwd}\n"
                f"Phone: {phoneno}\n"
                f"Address: {address}\n"
                f"Balance: Rs {balance}"

            ),
            justify="left",
            font=("Arial", 12),
            bg="white"
        ).pack(pady=5, anchor="w")

        tk.Button(frame,text="Edit",width=10,bg="#fbc02d",fg="white",
            command=lambda uid=user_id: edit_user_window(uid, win)
        ).pack(side="right", padx=5)

        tk.Button(frame,text="Delete",width=10,bg="#d32f2f",fg="white",
            command=lambda uid=user_id: [delete_user(uid), win.destroy(), view_users(parent)]
        ).pack(side="right", padx=5)

def edit_user_window(user_id, parent):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT name, email, password, phone_no, address, balance FROM users WHERE id=%s", (user_id,))
    user = cursor.fetchone()
    cursor.close()
    conn.close()

    if not user:
        messagebox.showerror("Error", "User not found!")
        return

    name_val, email_val, pwd_val, phone_val, address_val, balance_val = user

    win = tk.Toplevel(parent)
    win.title("Edit User")
    center_window(win, 1000, 600)
    win.configure(bg="#fff9c4")  

   
    tk.Label(win, text="Name:", bg="#fff9c4").pack(pady=5)
    name_entry = tk.Entry(win)
    name_entry.insert(0, str(name_val) if name_val else "")
    name_entry.pack()

 
    tk.Label(win, text="Email:", bg="#fff9c4").pack(pady=5)
    email_entry = tk.Entry(win)
    email_entry.insert(0,str(email_val) if email_val else "")
    email_entry.pack()

  
    tk.Label(win, text="Password:", bg="#fff9c4").pack(pady=5)
    pwd_entry = tk.Entry(win, show="*")
    pwd_entry.insert(0,str(pwd_val) if pwd_val else "")
    pwd_entry.pack()

    tk.Label(win, text="Phone No:", bg="#fff9c4").pack(pady=5)
    number_entry = tk.Entry(win)
    number_entry.insert(0,str(phone_val) if phone_val else "")
    number_entry.pack()
    
    tk.Label(win, text="Address:", bg="#fff9c4").pack(pady=5)
    address_entry = tk.Entry(win)
    address_entry.insert(0,  str(address_val) if address_val else "")
    address_entry.pack()
    
    tk.Label(win, text="Balance:", bg="#fff9c4").pack(pady=5)
    balance_entry = tk.Entry(win)
    balance_entry.pack()
    balance_entry.insert(0, balance_val)  


    def save_changes():
        edit_user(
            user_id,
            name_entry.get(),
            email_entry.get(),
            pwd_entry.get(),
            number_entry.get(),
            address_entry.get(),
            balance_entry.get()
        )
        messagebox.showinfo("Success", "User info updated!")
        win.destroy()
        admin_panel()  

    tk.Button(win, text="Save Changes", bg="#00796b", fg="white", command=save_changes).pack(pady=10)


def view_orders(parent):
    orders = fetch_orders()
    win = tk.Toplevel(parent)
    win.title("Orders")
    center_window(win, 1000, 600)
    win.configure(bg="#e0f7fa")

    tk.Label(win, text="Orders Placed by Users", font=("Helvetica", 20, "bold"), bg="#e0f7fa").pack(pady=10)

    container = tk.Frame(win, bg="#e0f7fa")
    container.pack(fill="both", expand=True)

    canvas = tk.Canvas(container, bg="#e0f7fa", highlightthickness=0)
    v_scroll = tk.Scrollbar(container, orient="vertical", command=canvas.yview)
    
    scroll_frame = tk.Frame(canvas, bg="#e0f7fa")

    scroll_frame.bind(
        "<Configure>",
        lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
    )
    canvas.create_window((0, 0), window=scroll_frame, anchor="nw")
    canvas.configure(yscrollcommand=v_scroll.set)

    canvas.pack(side="left", fill="both", expand=True)
    v_scroll.pack(side="right", fill="y")

    for order in orders:
        order_id, user_name, total, order_date, status = order
        frame = tk.Frame(scroll_frame, bg="white", bd=2, relief="groove")
        frame.pack(fill="x", padx=20, pady=5)

        tk.Label(frame, text=f"Order ID: {order_id} | User: {user_name} | Total: Rs {total} | Date: {order_date} | Status: {status}",
                 font=("Arial", 14), bg="white").pack(side="left", padx=10)
    
def choose_card_type(user_id, cart, total, parent):
    card_win = tk.Toplevel(parent)
    card_win.title("Choose Card Type")
    center_window(card_win, 400, 300)
    card_win.configure(bg="#f5f5f5")

    tk.Label(card_win, text="Select Card Type:", font=("Arial", 16, "bold"), bg="#f5f5f5").pack(pady=20)

    card_var = tk.StringVar(value="Standard")
    for card in CardPayment.CARD_DISCOUNTS.keys():
        tk.Radiobutton(card_win, text=f"{card} - {int(CardPayment.CARD_DISCOUNTS[card]*100)}% Discount",
                       variable=card_var, value=card, font=("Arial", 14), bg="#f5f5f5").pack(anchor="w", padx=20)

    def pay():
        card_win.destroy()
        CardPayment(user_id, cart, total, parent, card_type=card_var.get()).process_payment()

    tk.Button(card_win, text="Pay", font=("Arial", 14, "bold"), bg="#00796b", fg="white", command=pay).pack(pady=20)


def center_window(win, width, height):
    screen_width = win.winfo_screenwidth()
    screen_height = win.winfo_screenheight()
    x = (screen_width // 2) - (width // 2)
    y = (screen_height // 2) - (height // 2)
    win.geometry(f"{width}x{height}+{x}+{y}")
    win.resizable(False, False)

cart = []

def dashboard(username):
    dash = tk.Tk()
    dash.title("ATroves Dashboard")
    center_window(dash, 1200, 600)
    dash.configure(bg="#e0f7fa")

    nav_frame = tk.Frame(dash, bg="#0a1172", height=60)
    nav_frame.pack(fill="x")

    username_var = tk.StringVar(value=username)

    welcome_label = tk.Label(nav_frame, textvariable=username_var, font=("Arial", 16, "bold"),
                             bg="#0a1172", fg="white")
    welcome_label.pack(side="left", padx=20)

    content_frame = tk.Frame(dash, bg="#e0f7fa")
    content_frame.pack(fill="both", expand=True)
    
    def show_profile(parent):
        for widget in parent.winfo_children():
            widget.destroy()
            
        conn = get_connection()
        cursor = conn.cursor(dictionary=True)
        cursor.execute("SELECT id, name, email, phone_no, address FROM users WHERE name=%s", (username,))
        user = cursor.fetchone()
        cursor.close()
        conn.close()

        if not user:
            tk.Label(parent, text="User not found!", font=("Arial", 18, "bold"), fg="red", bg="#e0f7fa").pack(pady=20)
            return

        tk.Label(parent, text="Your Profile", font=("Arial", 22, "bold"), fg="#0a1172", bg="#e0f7fa").pack(pady=20)

        form_frame = tk.Frame(parent, bg="#e0f7fa")
        form_frame.pack(pady=10)

        tk.Label(form_frame, text="Name:", font=("Arial", 14), bg="#e0f7fa").grid(row=0, column=0, sticky="w", pady=5, padx=10)
        name_entry = tk.Entry(form_frame, font=("Arial", 14))
        name_entry.grid(row=0, column=1, pady=5, padx=10)
        name_entry.insert(0, user["name"])

        tk.Label(form_frame, text="Email:", font=("Arial", 14), bg="#e0f7fa").grid(row=1, column=0, sticky="w", pady=5, padx=10)
        email_entry = tk.Entry(form_frame, font=("Arial", 14))
        email_entry.grid(row=1, column=1, pady=5, padx=10)
        email_entry.insert(0, user["email"])

        tk.Label(form_frame, text="Phone No:", font=("Arial", 14), bg="#e0f7fa").grid(row=2, column=0, sticky="w", pady=5, padx=10)
        phone_entry = tk.Entry(form_frame, font=("Arial", 14))
        phone_entry.grid(row=2, column=1, pady=5, padx=10)
        phone_entry.insert(0, user["phone_no"] if user["phone_no"] else "")

        tk.Label(form_frame, text="Address:", font=("Arial", 14), bg="#e0f7fa").grid(row=3, column=0, sticky="w", pady=5, padx=10)
        address_entry = tk.Entry(form_frame, font=("Arial", 14), width=40)
        address_entry.grid(row=3, column=1, pady=5, padx=10)
        address_entry.insert(0, user["address"] if user["address"] else "")

        def update_user():
            new_name = name_entry.get()
            new_email = email_entry.get()
            new_phone = phone_entry.get()
            new_address = address_entry.get()

            if not new_name or not new_email:
                messagebox.showerror("Error", "Name and Email are required!")
                return

            conn = get_connection()
            cursor = conn.cursor()
            cursor.execute(
                "UPDATE users SET name=%s, email=%s, phone_no=%s, address=%s WHERE id=%s",
                (new_name, new_email, new_phone, new_address, user["id"])
            )
            conn.commit()
            cursor.close()
            conn.close()

            username_var.set(f"Welcome, {new_name}!")

            messagebox.showinfo("Success", "Profile updated successfully!")

        tk.Button(parent, text="Update", font=("Arial", 14, "bold"), bg="#00796b", fg="white",
                  command=update_user).pack(pady=20)
        
    def show_page(page):
        for widget in content_frame.winfo_children():
            widget.destroy()

        if page == "Catalog":
            build_catalog(content_frame)
        elif page == "Cart":
            build_cart(content_frame)
        elif page == "Profile":
            show_profile(content_frame) 
        elif page == "Logout":
            dash.destroy()
            main_menu()

    button_style = {"font": ("Arial", 12, "bold"), "bg": "#1565c0",
                    "fg": "white", "activebackground": "#003c8f", "width": 12}

    tk.Button(nav_frame, text="Catalog", command=lambda: show_page("Catalog"), **button_style).pack(side="left", padx=5)
    tk.Button(nav_frame, text="Cart", command=lambda: show_page("Cart"), **button_style).pack(side="left", padx=5)
    tk.Button(nav_frame, text="Profile", command=lambda: show_page("Profile"), **button_style).pack(side="left", padx=5)
    tk.Button(nav_frame, text="Logout", command=lambda: show_page("Logout"), **button_style).pack(side="right", padx=20)

    show_page("Catalog")
    dash.mainloop()

def build_catalog(parent):
    search_var = tk.StringVar()
    search_frame = tk.Frame(parent, bg="#e0f7fa")
    search_frame.pack(fill="x", pady=10)

    tk.Label(search_frame, text="Search:", font=("Arial", 14, "bold"),
             bg="#e0f7fa", fg="#0a1172").pack(side="left", padx=10)

    search_entry = tk.Entry(search_frame, textvariable=search_var, font=("Arial", 14), width=40)
    search_entry.pack(side="left", padx=10)
    
    canvas = tk.Canvas(parent, bg="#e0f7fa", highlightthickness=0)
    scrollbar = tk.Scrollbar(parent, orient="vertical", command=canvas.yview)
    scroll_frame = tk.Frame(canvas, bg="#e0f7fa")

    scroll_frame.bind(
        "<Configure>", 
        lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
    )

    window = canvas.create_window((0, 0), window=scroll_frame, anchor="nw")

    def resize_frame(event):
        canvas.itemconfig(window, width=event.width)

    canvas.bind("<Configure>", resize_frame)

    canvas.configure(yscrollcommand=scrollbar.set)

    canvas.pack(side="left", fill="both", expand=True)
    scrollbar.pack(side="right", fill="y")

    categories=fetch_categories()

    def update_catalog():
        query = search_var.get().lower()
        for widget in scroll_frame.winfo_children():
            widget.destroy()

        for cat_id, cat_name in categories:
            items = fetch_products_by_category(cat_id)
            cat_label = tk.Label(scroll_frame, text=cat_name, font=("Arial", 18, "bold"),
                                 bg="#81d4fa", fg="#0a1172", pady=5)
            cat_label.pack(fill="x", pady=5)

            for p in items:
                if query in p["name"].lower() or query == "":
                    frame = tk.Frame(scroll_frame, bg="white", bd=2, relief="groove")
                    frame.pack(fill="x", padx=20, pady=5)

                    tk.Label(frame, text=p["name"], font=("Arial", 14, "bold"), bg="white").pack(side="left", padx=10)
                    tk.Label(frame, text=f"Rs {p['price']}", font=("Arial", 12), fg="green", bg="white").pack(side="left", padx=10)

                    availability = "Available" if p["stock"] > 0 else "Out of Stock"
                    color = "green" if p["stock"] > 0 else "red"
                    tk.Label(frame, text=availability, font=("Arial", 12, "italic"),
                             fg=color, bg="white").pack(side="left", padx=10)

                    if p["stock"] > 0:
                        tk.Button(frame, text="Add to Cart", bg="#00796b", fg="white",
                                  command=lambda prod=p: add_to_cart(prod)).pack(side="right", padx=10)

    tk.Button(search_frame, text="Search", command=update_catalog,
              bg="#00796b", fg="white", font=("Arial", 12, "bold")).pack(side="left")
    search_entry.bind("<Return>", lambda e: update_catalog())
    update_catalog()

def update_stock(product_id, quantity):
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute(
            "UPDATE products SET stock = stock - %s WHERE id = %s AND stock >= %s",
            (quantity, product_id, quantity) 
        )
        conn.commit()
        return cursor.rowcount > 0  
    except Exception as e:
        print("Error while updating stock:", e)
        return False
    finally:
        cursor.close()
        conn.close()

def add_to_cart(product):
    conn = get_connection()
    cursor = conn.cursor(dictionary=True)

    cursor.execute("SELECT stock FROM products WHERE name = %s", (product["name"],))
    db_product = cursor.fetchone()

    cursor.close()
    conn.close()

    if not db_product:
        messagebox.showerror("Error", f"{product['name']} not found in DB!")
        return

    if db_product["stock"] <= 0:
        messagebox.showwarning("Out of Stock", f"{product['name']} is currently out of stock.")
        return

    for item in cart:
        if item["name"] == product["name"]:  
            if item.get("quantity", 1) < db_product["stock"]:
                item["quantity"] = item.get("quantity", 1) + 1
                messagebox.showinfo("Cart Updated", f"Added another {product['name']} to cart.")
            else:
                messagebox.showwarning("Out of Stock", f"No more {product['name']} left in stock.")
            return

    cart.append({
        "name": product["name"],
        "price": product["price"],
        "quantity": 1
    })
    messagebox.showinfo("Cart", f"{product['name']} added to cart!")

class Payment:
    def __init__(self, user_id, cart, total, parent):
        self.user_id = user_id
        self.cart = cart
        self.total = total
        self.parent = parent

    def process_payment(self):
        raise NotImplementedError("Subclasses must implement this method.")

    def generate_receipt(self):
        current_balance = get_user_balance(self.user_id)
        new_balance = current_balance - self.total if current_balance is not None else 0

        if isinstance(self, WalletPayment):
            update_user_balance(self.user_id, new_balance)
        
        save_order(self.user_id, self.cart, self.total)
    
        conn = get_connection()
        cursor = conn.cursor()
        try:
            for item in self.cart:
                qty = item.get("quantity", 1)
                cursor.execute(
                    "UPDATE products SET stock = stock - %s WHERE name = %s AND stock >= %s",
                    (qty, item["name"], qty)
                )
            conn.commit()
        finally:
            cursor.close()
            conn.close()

     
        bill_win = tk.Toplevel(self.parent)
        bill_win.title("ATroves Receipt")
        bill_win.geometry("500x500")
        bill_win.configure(bg="#f9f9f9")

        tk.Label(bill_win, text=" ATroves Receipt", font=("Arial", 20, "bold"),
                 fg="#1a237e", bg="#f9f9f9").pack(pady=20)

        for idx, item in enumerate(self.cart, start=1):
            qty = item.get("quantity", 1)
            tk.Label(bill_win, text=f"{idx}. {item['name']} (x{qty}) - Rs {item['price'] * qty}",
                     font=("Arial", 12),fg="#424242", bg="#f9f9f9").pack(anchor="w", padx=30)

        tk.Label(bill_win, text=f"Total Amount: Rs {self.total}", font=("Arial", 14, "bold"),
                 fg="#d32f2f",
                 bg="#f9f9f9").pack(pady=10)

        if isinstance(self, WalletPayment):
            tk.Label(bill_win, text=f"Previous Balance: Rs {current_balance}", font=("Arial", 12),
                     fg="#1565c0",bg="#f5f5f5").pack()
            tk.Label(bill_win, text=f"Remaining Balance: Rs {new_balance}", font=("Arial", 12, "bold"),
                     fg="#2e7d32", bg="#f9f9f9").pack()

        tk.Button(bill_win, text="Close", command=bill_win.destroy,
                  font=("Arial", 12, "bold"), bg="#0d47a1", fg="white",
                  activebackground="#1565c0",activeforeground="white",
                  relief="flat", padx=20, pady=5).pack(pady=20)

        self.cart.clear()


class CashPayment(Payment):
    def process_payment(self):
        self.generate_receipt()


class WalletPayment(Payment):
    def process_payment(self):
        current_balance = get_user_balance(self.user_id)
        if current_balance is None:
            messagebox.showerror("Error", "User not found in DB!")
            return
        if self.total > current_balance:
            messagebox.showerror("Insufficient Balance", "Not enough balance!")
            return
        self.generate_receipt()


class CardPayment(Payment):
    CARD_DISCOUNTS = {
        "HBL": Decimal("0.10"),
        "MCB": Decimal("0.05"),
        "Standard": Decimal("0.0")
    }

    def __init__(self, user_id, cart, total, parent, card_type="Standard"):
        super().__init__(user_id, cart, total, parent)
        self.card_type = card_type if card_type in self.CARD_DISCOUNTS else "Standard"
        if not isinstance(self.total, Decimal):
            self.total = Decimal(str(self.total))

    def process_payment(self):
        discount = self.CARD_DISCOUNTS.get(self.card_type, Decimal("0.0"))
        discounted_total = self.total * (Decimal("1.0") - discount)
        self.total = discounted_total.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        self.generate_receipt()


def build_cart(parent):
    canvas = tk.Canvas(parent, bg="#e0f7fa", highlightthickness=0)
    scrollbar = tk.Scrollbar(parent, orient="vertical", command=canvas.yview)
    scroll_frame = tk.Frame(canvas, bg="#e0f7fa")

    scroll_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
    canvas.create_window((0, 0), window=scroll_frame, anchor="nw")
    canvas.configure(yscrollcommand=scrollbar.set)

    canvas.pack(side="left", fill="both", expand=True)
    scrollbar.pack(side="right", fill="y")

    if not cart:
        tk.Label(scroll_frame, text="Your cart is empty.", font=("Arial", 18, "bold"),
                 fg="red", bg="#e0f7fa").pack(pady=20)
        return

    tk.Label(scroll_frame, text="Your Shopping Cart", font=("Arial", 22, "bold"),
             fg="#0a1172", bg="#e0f7fa").pack(pady=20)

    total = 0
    for idx, item in enumerate(cart, start=1):
        frame = tk.Frame(scroll_frame, bg="white", bd=2, relief="groove")
        frame.pack(fill="x", padx=20, pady=5)

        qty = item.get("quantity", 1)
        line_total = item["price"] * qty
        total += line_total

        tk.Label(frame, text=f"{idx}. {item['name']} (x{qty})", font=("Arial", 14, "bold"), bg="white").pack(side="left", padx=10)
        tk.Label(frame, text=f"Rs {line_total}", font=("Arial", 12), fg="green", bg="white").pack(side="left", padx=10)

    tk.Label(scroll_frame, text=f"Total: Rs {total}", font=("Arial", 18, "bold"),
             fg="green", bg="#e0f7fa").pack(pady=20)

    def generate_bill(total=total):
        global logged_in_user
        user_id = logged_in_user["id"]

        current_balance = get_user_balance(user_id)
        if current_balance is None:
            messagebox.showerror("Error", "User not found in DB!")
            return

        if total > current_balance:
            messagebox.showerror("Insufficient Balance", "Not enough balance to complete purchase.")
            return

        new_balance = current_balance - total
        update_user_balance(user_id, new_balance)
        save_order(user_id, cart, total)

        conn = get_connection()
        cursor = conn.cursor()
        try:
            for item in cart:
                qty = item.get("quantity", 1)
                cursor.execute(
                    "UPDATE products SET stock = stock - %s WHERE name = %s AND stock >= %s",
                    (qty, item["name"], qty)
                )
            conn.commit()
        finally:
            cursor.close()
            conn.close()

        bill_win = tk.Toplevel(parent)
        bill_win.title("ATroves Receipt")
        bill_win.geometry("500x500")
        bill_win.configure(bg="#f9f9f9")

        tk.Label(bill_win, text=" ATroves Receipt", font=("Arial", 20, "bold"),
                 fg="#1a237e", bg="#f9f9f9").pack(pady=20)

        for idx, item in enumerate(cart, start=1):
            qty = item.get("quantity", 1)
            tk.Label(bill_win, text=f"{idx}. {item['name']} (x{qty}) - Rs {item['price'] * qty}",
                     font=("Arial", 12),fg="#424242", bg="#f9f9f9").pack(anchor="w", padx=30)

        tk.Label(bill_win, text=f"Total Amount: Rs {total}", font=("Arial", 14, "bold"),
                 fg="#d32f2f",
                 bg="#f9f9f9").pack(pady=10)

        tk.Label(bill_win, text=f"Previous Balance: Rs {current_balance}", font=("Arial", 12),
                 fg="#1565c0",bg="#f5f5f5").pack()
        tk.Label(bill_win, text=f"Remaining Balance: Rs {new_balance}", font=("Arial", 12, "bold"),
                 fg="#2e7d32", bg="#f9f9f9").pack()

        tk.Button(bill_win, text="Close", command=bill_win.destroy,
                  font=("Arial", 12, "bold"), bg="#0d47a1", fg="white",activebackground="#1565c0",activeforeground="white",relief="flat",padx=20,pady=5).pack(pady=20)

        cart.clear()

    btn_frame = tk.Frame(scroll_frame, bg="#e0f7fa")
    btn_frame.pack(fill="x", pady=10)

    btn_frame = tk.Frame(scroll_frame, bg="#e0f7fa")
    btn_frame.pack(fill="x", pady=10)

    user_id = logged_in_user["id"]

    tk.Button(btn_frame, text="Pay by Cash", font=("Arial", 14, "bold"),
              bg="#00796b", fg="white", width=20, height=2,
              command=lambda: CashPayment(user_id, cart, total, parent).process_payment()).pack(pady=5)

    tk.Button(btn_frame, text="Pay by Wallet", font=("Arial", 14, "bold"),
              bg="#FFA000", fg="white", width=20, height=2,
              command=lambda: WalletPayment(user_id, cart, total, parent).process_payment()).pack(pady=5)
    tk.Button(btn_frame, text="Pay by Card", font=("Arial", 14, "bold"),
              bg="#1976d2", fg="white", width=20, height=2,
              command=lambda: choose_card_type(user_id, cart, total, parent)).pack(pady=5)

    

def center_window(win, width, height):
    screen_width = win.winfo_screenwidth()
    screen_height = win.winfo_screenheight()
    x = (screen_width // 2) - (width // 2)
    y = (screen_height // 2) - (height // 2)
    win.geometry(f"{width}x{height}+{x}+{y}")
    win.resizable(False, False)

cart = []


def front_screen():
    front = tk.Tk()
    front.title("ATroves Store")
    center_window(front, 1300, 785)
    front.configure(bg="#7AC142")

    try:
        image_path = "13.jpg"
        img = Image.open(image_path)
        img = img.resize((1290, 785))
        logo = ImageTk.PhotoImage(img)

        logo_label = tk.Label(front, image=logo, bg="white")
        logo_label.image = logo
        logo_label.place(relx=0.5, rely=0.5, anchor="center")
    except Exception as e:
        print("Error loading image:", e)
        tk.Label(front, text="ATroves Online Store",
                 font=("Helvetica", 32, "bold"), bg="#0a1172", fg="#ffde59").pack(pady=100)

    def go_to_menu():
        front.destroy()
        main_menu()
    front.after(2500, go_to_menu)
    front.mainloop()

def main_menu():
    menu = tk.Tk()
    menu.title("Main Menu")
    center_window(menu, 1300, 785)
    menu.configure(bg="#e0f7fa")

    tk.Label(menu, text="Welcome to ATROVES Store", font=("Verdana", 28, "bold"), bg="#e0f7fa", fg="#0a1172").pack(pady=30)
    tk.Label(menu, text="Select an Option", font=("Calibri", 20, "bold"), bg="#e0f7fa", fg="#00796b").pack(pady=10)

    def open_login():
        menu.destroy()
        login_window()

    def open_signup():
        menu.destroy()
        signup_window()

    def open_admin():
        menu.destroy()
        admin_window()

    def open_goodbye():
        menu.destroy()
        goodbye_window()

    button_style = {"font": ("Arial", 16, "bold"), "width": 18, "bg": "#2E7D32", "fg": "white", "activebackground": "#006064"}

    tk.Button(menu, text="Login", command=open_login, **button_style).pack(pady=10)
    tk.Button(menu, text="Signup", command=open_signup, **button_style).pack(pady=10)
    tk.Button(menu, text="Admin", command=open_admin, **button_style).pack(pady=10)
    tk.Button(menu, text="Exit", command=open_goodbye, **button_style).pack(pady=10)

    menu.mainloop()

def login_window():
    login = tk.Tk()
    login.title("Login")
    center_window(login, 1300, 785)
    login.configure(bg="#e0f2f1")

    tk.Label(login, text="Login to Your Account", font=("Helvetica", 24, "bold"),
             bg="#e0f2f1", fg="#004d40").pack(pady=30)

    tk.Label(login, text="Username:", bg="#e0f2f1", font=("Arial", 16)).pack(pady=5)
    username_entry = tk.Entry(login, font=("Arial", 14))
    username_entry.pack(pady=5)

    tk.Label(login, text="Password:", bg="#e0f2f1", font=("Arial", 16)).pack(pady=5)
    password_entry = tk.Entry(login, show="*", font=("Arial", 14))
    password_entry.pack(pady=5)

    def validate_login():
        global logged_in_user  

        if username_entry.get() == "" or password_entry.get() == "":
            messagebox.showerror("Error", "All fields are required!")
            return

        conn = get_connection()
        cursor = conn.cursor(dictionary=True)
        cursor.execute("SELECT * FROM users WHERE name=%s AND password=%s",
                       (username_entry.get(), password_entry.get()))
        logged_in_user = cursor.fetchone()
        cursor.close()
        conn.close()

        if logged_in_user:
            messagebox.showinfo("Success", "Login Successful!")
            login.destroy()
            dashboard(logged_in_user["name"]) 
        else:
            messagebox.showerror("Error", "Invalid username or password!")

    def back_to_menu():
        login.destroy()
        main_menu()

    tk.Button(login, text="Login", font=("Arial", 15, "bold"),
              command=validate_login, bg="#00796b", fg="white").pack(pady=10)
    tk.Button(login, text="Back", font=("Arial", 15, "bold"),
              command=back_to_menu, bg="#b71c1c", fg="white").pack()

    login.mainloop()


def signup_window():
    signup = tk.Tk()
    signup.title("Signup")
    center_window(signup, 1300, 785)
    signup.configure(bg="#fce4ec")

    tk.Label(signup, text="Create a New Account",
             font=("Helvetica", 24, "bold"),
             bg="#fce4ec", fg="#880e4f").pack(pady=30)

    tk.Label(signup, text="Username:", bg="#fce4ec", font=("Arial", 15)).pack(pady=5)
    username_entry = tk.Entry(signup, font=("Arial", 14))
    username_entry.pack()

    tk.Label(signup, text="Password:", bg="#fce4ec", font=("Arial", 15)).pack(pady=5)
    password_entry = tk.Entry(signup, show="*", font=("Arial", 14))
    password_entry.pack()

    tk.Label(signup, text="Confirm Password:", bg="#fce4ec", font=("Arial", 15)).pack(pady=5)
    confirm_entry = tk.Entry(signup, show="*", font=("Arial", 14))
    confirm_entry.pack()

    tk.Label(signup, text="Phone No:", bg="#fce4ec", font=("Arial", 15)).pack(pady=5)
    number_entry = tk.Entry(signup, font=("Arial", 14))
    number_entry.pack()

    tk.Label(signup, text="Email ID:", bg="#fce4ec", font=("Arial", 15)).pack(pady=5)
    email_entry = tk.Entry(signup, font=("Arial", 14))
    email_entry.pack()

    tk.Label(signup, text="Address:", bg="#fce4ec", font=("Arial", 15)).pack(pady=5)
    address_entry = tk.Entry(signup, font=("Arial", 14))
    address_entry.pack()

    def validate_signup():
        username = username_entry.get().strip()
        password = password_entry.get().strip()
        confirm = confirm_entry.get().strip()
        phone = number_entry.get().strip()
        email = email_entry.get().strip()
        address = address_entry.get().strip()

        if not (username and password and confirm and phone and email and address):
            messagebox.showerror("Error", "All fields are required!")
            return

        if password != confirm:
            messagebox.showerror("Error", "Passwords do not match!")
            return

        if not (phone.isdigit() and len(phone) == 11):
            messagebox.showerror("Error", "Phone number must be exactly 11 digits!")
            return
        
        if not email.endswith("@gmail.com"):
            messagebox.showerror("Error", "Email must end with @gmail.com")
            return    

        if save_user(username, password, phone, address, email):
            messagebox.showinfo("Success", "Signup Successful! You can now login.")
            signup.destroy()
            login_window()
        else:
            messagebox.showerror("Error", "Username already exists! Try another.")

    tk.Button(signup, text="Signup", font=("Arial", 14, "bold"),
              bg="#880e4f", fg="white", width=20, command=validate_signup).pack(pady=20)

    tk.Button(signup, text="Back to Login", font=("Arial", 14, "bold"),
              bg="#1565c0", fg="white", width=20,
              command=lambda: [signup.destroy(), login_window()]).pack(pady=10)

    signup.mainloop()

def manage_stock(parent):
    win = tk.Toplevel(parent)
    win.title("Manage Stock")
    center_window(win, 1000, 600)
    win.configure(bg="#e0f7fa")

    tk.Label(win, text="Manage Product Stock", font=("Arial", 22, "bold"),
             fg="#0a1172", bg="#e0f7fa").pack(pady=20)

    conn = get_connection()
    cursor = conn.cursor(dictionary=True)
    cursor.execute("SELECT id, name, stock FROM products")
    products = cursor.fetchall()
    cursor.close()
    conn.close()

    container = tk.Frame(win, bg="#e0f7fa")
    container.pack(fill="both", expand=True, padx=20, pady=10)

    canvas = tk.Canvas(container, bg="#e0f7fa", highlightthickness=0)
    v_scroll = tk.Scrollbar(container, orient="vertical", command=canvas.yview)

    table_frame = tk.Frame(canvas, bg="#e0f7fa")

    table_frame.bind(
        "<Configure>",
        lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
    )

    canvas.create_window((0, 0), window=table_frame, anchor="nw")
    canvas.configure(yscrollcommand=v_scroll.set)

    canvas.pack(side="left", fill="both", expand=True)
    v_scroll.pack(side="right", fill="y")

    tk.Label(table_frame, text="Product", font=("Arial", 14, "bold"),
             bg="#e0f7fa").grid(row=0, column=0, padx=10, pady=5)
    tk.Label(table_frame, text="Stock", font=("Arial", 14, "bold"),
             bg="#e0f7fa").grid(row=0, column=1, padx=10, pady=5)
    tk.Label(table_frame, text="Action", font=("Arial", 14, "bold"),
             bg="#e0f7fa").grid(row=0, column=2, padx=10, pady=5)

    row = 1
    for product in products:
        tk.Label(table_frame, text=product["name"], font=("Arial", 12),
                 bg="#e0f7fa").grid(row=row, column=0, padx=10, pady=5)

        stock_var = tk.StringVar(value=str(product["stock"]))
        stock_entry = tk.Entry(table_frame, textvariable=stock_var,
                               font=("Arial", 12), width=10)
        stock_entry.grid(row=row, column=1, padx=10, pady=5)

        def save_stock(p_id=product["id"], stock_var=stock_var):
            new_stock = stock_var.get()
            if not new_stock.isdigit():
                messagebox.showerror("Error", "Stock must be a number!")
                return
            conn = get_connection()
            cursor = conn.cursor()
            cursor.execute("UPDATE products SET stock=%s WHERE id=%s",
                           (int(new_stock), p_id))
            conn.commit()
            cursor.close()
            conn.close()
            messagebox.showinfo("Success", f"Stock updated for product ID {p_id}")

        tk.Button(table_frame, text="Update", font=("Arial", 12),
                  bg="#00796b", fg="white", command=save_stock).grid(row=row, column=2, padx=10, pady=5)

        row += 1
        
    tk.Button(win, text="Back", font=("Arial", 14, "bold"),
              bg="#b71c1c", fg="white", width=15,
              command=win.destroy).pack(pady=15)



def admin_panel():
    panel = tk.Tk()
    panel.title("Admin Panel")
    center_window(panel, 1300, 785)
    panel.configure(bg="#e3f2fd")

    tk.Label(panel, text="Admin Panel", font=("Helvetica", 24, "bold"), bg="#e3f2fd", fg="#0d47a1").pack(pady=20)

    button_frame = tk.Frame(panel, bg="#e3f2fd")
    button_frame.pack(pady=10)

    tk.Button(button_frame, text="View All Users", font=("Arial", 14, "bold"),
              bg="#1976d2", fg="white", width=18, command=lambda: view_users(panel)).grid(row=0, column=0, padx=10, pady=10)

    tk.Button(button_frame, text="View Orders", font=("Arial", 14, "bold"),
              bg="#00796b", fg="white", width=18, command=lambda: view_orders(panel)).grid(row=0, column=1, padx=10, pady=10)

    tk.Button(button_frame, text="Manage Stock", font=("Arial", 14, "bold"),
              bg="#1976d2", fg="white", width=18, command=lambda: manage_stock(panel)).grid(row=0, column=2, padx=10, pady=10)

    tk.Button(button_frame, text="Back", font=("Arial", 14, "bold"),
              bg="#b71c1c", fg="white", width=18, command=lambda: [panel.destroy(), main_menu()]).grid(row=0, column=3, padx=10, pady=10)

    panel.mainloop()



def admin_window():
    admin = tk.Tk()
    admin.title("Admin Login")
    center_window(admin, 1300, 785)
    admin.configure(bg="#e3f2fd")

    tk.Label(admin, text="Admin Login", font=("Helvetica", 24, "bold"), bg="#e3f2fd", fg="#0d47a1").pack(pady=30)

    tk.Label(admin, text="Admin Username:", bg="#e3f2fd", font=("Arial", 15)).pack(pady=5)
    username_entry = tk.Entry(admin, font=("Arial", 14))
    username_entry.pack()

    tk.Label(admin, text="Password:", bg="#e3f2fd", font=("Arial", 15)).pack(pady=5)
    password_entry = tk.Entry(admin, show="*", font=("Arial", 14))
    password_entry.pack()

    def validate_admin():
        if username_entry.get() == "" or password_entry.get() == "":
            messagebox.showerror("Error", "All fields are required!")
            return
        if check_admin(username_entry.get(), password_entry.get()):
            messagebox.showinfo("Success", "Welcome Admin!")
            admin.destroy()
            admin_panel()
        else:
            messagebox.showerror("Error", "Invalid admin credentials!")

    def back_to_menu():
        admin.destroy()
        main_menu()

    tk.Button(admin, text="Login", font=("Arial", 15, "bold"), command=validate_admin, bg="#1976d2", fg="white").pack(pady=10)
    tk.Button(admin, text="Back", font=("Arial", 15, "bold"), command=back_to_menu, bg="#b71c1c", fg="white").pack()

    admin.mainloop()

def goodbye_window():
    bye = tk.Tk()
    bye.title("Goodbye")
    center_window(bye, 1300, 785)
    bye.configure(bg="#7AC142")
    
    try:
        image_path = "12.png"
        img = Image.open(image_path)
        img = img.resize((400, 400))
        goodbye_img = ImageTk.PhotoImage(img)

        lbl = tk.Label(bye, image=goodbye_img, bg="#7AC142")
        lbl.image = goodbye_img
        lbl.place(relx=0.5, rely=0.5, anchor="center")
    except Exception as e:
        print("Error loading image:", e)
        tk.Label(bye, text="Goodbye! See you again!",
                 font=("Arial", 30, "bold"), fg="#ff6f00", bg="#ffecb3").pack(expand=True)
    bye.after(2000, bye.destroy)
    bye.mainloop()
 
front_screen()
    
